In [1]:
!git clone https://github.com/baudm/parseq.git --depth 1
!pip install -q timm einops omegaconf pytorch-lightning ultralytics
!pip install -q hydra-core
import sys
sys.path.append("/kaggle/working/parseq")

Cloning into 'parseq'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 127 (delta 9), reused 63 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 697.70 KiB | 5.33 MiB/s, done.
Resolving deltas: 100% (9/9), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.6 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.

In [2]:
%%writefile inference_optimized.py
import os
import sys

# ============================================================
# CRITICAL: GPU ISOLATION & NETWORK FIX FOR KAGGLE DOCKER
# ============================================================
if "LOCAL_RANK" in os.environ:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(os.environ["LOCAL_RANK"])
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["NCCL_SOCKET_IFNAME"] = "lo"
os.environ["NCCL_P2P_DISABLE"] = "1"

sys.path.append("/kaggle/working/parseq")

import glob
import cv2
import torch
import pandas as pd
import torch.distributed as dist
import concurrent.futures

from tqdm.auto import tqdm
from ultralytics import YOLO

# ============================================================
# CẤU HÌNH TỐI ƯU CUDA
# ============================================================
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cuda.matmul.allow_tf32 = True

# ============================================================
# ĐƯỜNG DẪN DỮ LIỆU
# ============================================================
DATASET_DIR = "/kaggle/input/datasets/vdt1501/license-plate-detection-dataset/License Plate Detection Dataset/images/test"
YOLO_WEIGHT = "/kaggle/input/datasets/vdt1501/yolo-parseq/best.pt"
PARSEQ_WEIGHT = "/kaggle/input/datasets/vdt1501/parseq-bb5792a6-pt/parseq-bb5792a6.pt"
OUTPUT_CSV = "/kaggle/working/license_plate_results.csv"

from strhub.models.utils import create_model


# ============================================================
# HÀM HỖ TRỢ
# ============================================================
def setup_ddp():
    """Khởi tạo môi trường Distributed Data Parallel."""
    dist.init_process_group(backend="gloo")
    
    local_rank = int(os.environ["LOCAL_RANK"])
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    
    torch.cuda.set_device(0) 
    return local_rank, rank, world_size


def load_image(path):
    """Đọc ảnh bằng cv2 (được gọi đa luồng)."""
    return path, cv2.imread(path)


def fast_preprocess(crop):
    """Tiền xử lý ảnh cắt (crop) trực tiếp bằng cv2 + numpy."""
    rgb = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    resized = cv2.resize(rgb, (128, 32), interpolation=cv2.INTER_LINEAR)
    tensor = torch.from_numpy(resized).permute(2, 0, 1).float() / 127.5 - 1.0
    return tensor


def batch_ocr(crops, parseq_model, device, ocr_batch_size):
    """Thực hiện OCR với CPU đa luồng cho preprocessing + FP16 cho inference."""
    tensors = []

    if len(crops) > 0:
        num_workers = min(os.cpu_count() or 4, len(crops))
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
            tensors = list(executor.map(fast_preprocess, crops))

    texts_all = []
    for i in range(0, len(tensors), ocr_batch_size):
        batch = torch.stack(tensors[i:i + ocr_batch_size]).to(device, non_blocking=True)

        with torch.inference_mode():
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                logits = parseq_model(batch)
                probs = logits.softmax(-1)

        texts, _ = parseq_model.tokenizer.decode(probs)
        texts_all.extend(texts)

    return texts_all


# ============================================================
# HÀM CHÍNH
# ============================================================
def main():
    local_rank, rank, world_size = setup_ddp()
    device = torch.device("cuda:0")
    
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    num_cpu_cores = os.cpu_count() or 4

    # --------------------------------------------------------
    # 1. CẤU HÌNH BATCH SIZE & IMAGE SIZE
    # --------------------------------------------------------
    if vram_gb > 20:  # GPU 30GB
        YOLO_BATCH_LOCAL = 128
        YOLO_IMGSZ = 640
        OCR_BATCH_LOCAL = 8192
    else:             # GPU T4 15GB
        # GIẢM IMGSZ TỪ 640 XUỐNG 416 ĐỂ TIẾT KIỆM ~50% VRAM
        # Batch 16 ở imgsz 416 chỉ tốn ~4-5 GB VRAM
        YOLO_BATCH_LOCAL = 16
        YOLO_IMGSZ = 416
        OCR_BATCH_LOCAL = 4096

    if rank == 0:
        print("=" * 60)
        print(f"[MASTER] World size: {world_size} GPUs")
        print(f"[MASTER] CPU cores : {num_cpu_cores}")
        print("=" * 60)
    print(f"[GPU {local_rank}] {gpu_name} | VRAM: {vram_gb:.1f} GB | "
          f"YOLO_BATCH={YOLO_BATCH_LOCAL} | IMGSZ={YOLO_IMGSZ} | OCR_BATCH={OCR_BATCH_LOCAL}")

    # --------------------------------------------------------
    # 2. LOAD MODELS
    # --------------------------------------------------------
    detector = YOLO(YOLO_WEIGHT)

    parseq = create_model("parseq", pretrained=False)
    state_dict = torch.load(PARSEQ_WEIGHT, map_location="cpu")
    state_dict = {"model." + k: v for k, v in state_dict.items()}
    parseq.load_state_dict(state_dict)
    parseq = parseq.to(device).eval()

    # --------------------------------------------------------
    # 3. PHÂN VÙNG DỮ LIỆU
    # --------------------------------------------------------
    image_files = sorted([
        os.path.join(DATASET_DIR, f)
        for f in os.listdir(DATASET_DIR)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
    my_files = image_files[rank::world_size]

    if rank == 0:
        print(f"[MASTER] Total images: {len(image_files)} | "
              f"Per GPU: ~{len(my_files)}")

    # --------------------------------------------------------
    # 4. ĐỌC ẢNH VÀO RAM BẰNG CPU ĐA LUỒNG
    # --------------------------------------------------------
    image_cache = {}
    if len(my_files) > 0:
        with concurrent.futures.ThreadPoolExecutor(max_workers=num_cpu_cores) as executor:
            for path, img in executor.map(load_image, my_files):
                image_cache[path] = img

    # --------------------------------------------------------
    # 5. YOLO DETECTION
    # --------------------------------------------------------
    torch.cuda.empty_cache()
    
    preds = detector.predict(
        source=my_files,
        imgsz=YOLO_IMGSZ,  # SỬ DỤNG IMGSZ ĐỘNG (416 cho T4, 640 cho GPU 30GB)
        conf=0.20,
        iou=0.50,
        batch=YOLO_BATCH_LOCAL,
        device=0,
        half=True,
        verbose=(rank == 0),
        stream=False
    )

    # --------------------------------------------------------
    # 6. OCR LOOP
    # --------------------------------------------------------
    results_all = []
    iterator = list(zip(my_files, preds))
    if rank == 0:
        iterator = tqdm(iterator, desc=f"[GPU {local_rank}] OCR")

    for img_path, pred in iterator:
        image = image_cache.get(img_path)
        if image is None:
            continue

        h, w = image.shape[:2]
        boxes = pred.boxes.xyxy.cpu().numpy()
        
        # Scale boxes về kích thước gốc nếu imgsz != original size
        # Ultralytics tự động xử lý việc này trong pred.boxes
        
        if len(boxes) == 0:
            continue

        crops, coords = [], []
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            pad = 5
            x1 = max(0, x1 - pad)
            y1 = max(0, y1 - pad)
            x2 = min(w, x2 + pad)
            y2 = min(h, y2 + pad)
            crops.append(image[y1:y2, x1:x2])
            coords.append([x1, y1, x2, y2])

        try:
            texts = batch_ocr(crops, parseq, device, OCR_BATCH_LOCAL)
        except Exception as e:
            if rank == 0:
                print(f"[ERROR] OCR failed: {e}")
            continue

        for coord, text in zip(coords, texts):
            results_all.append({
                "image": os.path.basename(img_path),
                "plate": text,
                "bbox": coord
            })

    # --------------------------------------------------------
    # 7. LƯU KẾT QUẢ
    # --------------------------------------------------------
    out_csv_rank = OUTPUT_CSV.replace(".csv", f"_part{rank}.csv")
    pd.DataFrame(results_all).to_csv(out_csv_rank, index=False)
    print(f"[GPU {local_rank}] Saved {len(results_all)} plates -> {out_csv_rank}")

    # --------------------------------------------------------
    # 8. ĐỒNG BỘ VÀ GỘP KẾT QUẢ
    # --------------------------------------------------------
    dist.barrier()

    if rank == 0:
        part_files = sorted(glob.glob(OUTPUT_CSV.replace(".csv", "_part*.csv")))
        if part_files:
            df_combined = pd.concat(
                [pd.read_csv(f) for f in part_files],
                ignore_index=True
            )
            df_combined.to_csv(OUTPUT_CSV, index=False)
            print("=" * 60)
            print(f"[MASTER] Final CSV: {OUTPUT_CSV}")
            print(f"[MASTER] Total plates: {len(df_combined)}")
            print(f"[MASTER] File size: {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB")
            print("=" * 60)

    dist.destroy_process_group()


if __name__ == "__main__":
    main()

Writing inference_optimized.py


In [3]:
!torchrun --nproc_per_node=2 --standalone inference_optimized.py

W0624 15:30:38.581000 128 torch/distributed/run.py:852] 
W0624 15:30:38.581000 128 torch/distributed/run.py:852] *****************************************
W0624 15:30:38.581000 128 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0624 15:30:38.581000 128 torch/distributed/run.py:852] *****************************************
[W624 15:30:38.319967199 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
WARNING ⚠️ Error decoding JSON from /root/.config/Ultralytics/settings.json. Starting with an empty dictionary.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://doc

Đánh giá mà ko có groudtruth

In [4]:
%%writefile evaluate_no_gt.py
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

def evaluate_ocr_no_gt(csv_path, image_dir, output_dir='/kaggle/working/eval_no_gt'):
    """
    Đánh giá chất lượng OCR predictions mà không cần ground truth.
    """
    os.makedirs(output_dir, exist_ok=True)
    
    print("="*80)
    print("🚀 BẮT ĐẦU ĐÁNH GIÁ OCR (KHÔNG CẦN GROUND TRUTH)")
    print("="*80)
    
    # Load predictions
    print(f"\n📂 Loading predictions từ: {csv_path}")
    try:
        df = pd.read_csv(csv_path)
        print(f"   ✅ Loaded {len(df)} predictions")
    except Exception as e:
        print(f"   ❌ Lỗi khi load file: {e}")
        return None
    
    # Parse bbox
    def parse_bbox(bbox_str):
        try:
            if isinstance(bbox_str, str):
                bbox_str = bbox_str.strip()
                if bbox_str.startswith('[') and bbox_str.endswith(']'):
                    return eval(bbox_str)
                elif bbox_str.startswith('(') and bbox_str.endswith(')'):
                    return eval(bbox_str)
            elif isinstance(bbox_str, (list, tuple)):
                return list(bbox_str)
            return None
        except:
            return None
    
    df['bbox_parsed'] = df['bbox'].apply(parse_bbox)
    valid_mask = df['bbox_parsed'].apply(lambda x: x is not None and len(x) == 4)
    df = df[valid_mask].copy()
    
    print(f"   ✅ {len(df)} predictions có bbox hợp lệ")
    
    if len(df) == 0:
        print("   ❌ Không có predictions hợp lệ!")
        return None
    
    # Tính features
    df['bbox_width'] = df['bbox_parsed'].apply(lambda x: abs(x[2] - x[0]))
    df['bbox_height'] = df['bbox_parsed'].apply(lambda x: abs(x[3] - x[1]))
    df['bbox_area'] = df['bbox_width'] * df['bbox_height']
    df['aspect_ratio'] = df['bbox_width'] / (df['bbox_height'] + 1e-6)
    df['plate'] = df['plate'].astype(str).str.upper().str.strip()
    df['plate_length'] = df['plate'].str.len()
    
    # Tính reliability score
    print("\n🔢 Đang tính reliability score...")
    
    reliability_scores = []
    
    for idx, row in df.iterrows():
        score = 100
        issues = []
        
        plate = row['plate']
        
        # Kiểm tra độ dài
        if row['plate_length'] < 3:
            score -= 30
            issues.append('Too short')
        elif row['plate_length'] > 10:
            score -= 20
            issues.append('Too long')
        elif 6 <= row['plate_length'] <= 9:
            score += 10
            issues.append('Good length')
        
        # Kiểm tra ký tự
        if not re.match(r'^[A-Z0-9]+$', plate):
            score -= 40
            issues.append('Invalid characters')
        
        # Kiểm tra aspect ratio
        if row['aspect_ratio'] < 2:
            score -= 20
            issues.append('Too square')
        elif row['aspect_ratio'] > 7:
            score -= 15
            issues.append('Too elongated')
        elif 3 <= row['aspect_ratio'] <= 5:
            score += 10
            issues.append('Good aspect ratio')
        
        # Kiểm tra diện tích
        if row['bbox_area'] < 1000:
            score -= 25
            issues.append('Very small')
        elif row['bbox_area'] > 50000:
            score -= 10
            issues.append('Very large')
        
        reliability_scores.append({
            'image': row['image'],
            'plate': plate,
            'reliability_score': max(0, score),
            'issues': ', '.join(issues) if issues else 'None',
            'bbox_area': row['bbox_area'],
            'aspect_ratio': row['aspect_ratio'],
            'plate_length': row['plate_length'],
            'bbox': row['bbox_parsed']
        })
    
    df_result = pd.DataFrame(reliability_scores)
    
    print(f"   ✅ Đã tính score cho {len(df_result)} predictions")
    
    # Thống kê
    print("\n" + "="*80)
    print("📊 KẾT QUẢ ĐÁNH GIÁ")
    print("="*80)
    
    avg_score = df_result['reliability_score'].mean()
    median_score = df_result['reliability_score'].median()
    
    high_conf = df_result[df_result['reliability_score'] >= 80]
    medium_conf = df_result[(df_result['reliability_score'] >= 60) & 
                           (df_result['reliability_score'] < 80)]
    low_conf = df_result[df_result['reliability_score'] < 60]
    
    print(f"\n📈 TỔNG QUAN:")
    print(f"  • Tổng số predictions: {len(df_result)}")
    print(f"  • Điểm trung bình: {avg_score:.1f}/100")
    print(f"  • Điểm trung vị: {median_score:.1f}/100")
    
    print(f"\n🎯 PHÂN LOẠI ĐỘ TIN CẬY:")
    print(f"  • High confidence (≥80): {len(high_conf):4d} ({len(high_conf)/len(df_result)*100:.1f}%)")
    print(f"  • Medium confidence (60-79): {len(medium_conf):4d} ({len(medium_conf)/len(df_result)*100:.1f}%)")
    print(f"  • Low confidence (<60): {len(low_conf):4d} ({len(low_conf)/len(df_result)*100:.1f}%)")
    
    # Phân tích issues
    all_issues = []
    for issues in df_result['issues']:
        if issues != 'None':
            all_issues.extend([i.strip() for i in issues.split(',')])
    
    if all_issues:
        issue_counts = Counter(all_issues)
        print(f"\n⚠️  CÁC VẤN ĐỀ PHỔ BIẾN:")
        for issue, count in issue_counts.most_common(5):
            print(f"    • {issue:30s}: {count:3d} lần")
    
    print("="*80)
    
    # Trực quan hóa
    print("\n📊 Đang tạo biểu đồ...")
    
    try:
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('🔍 OCR Quality Assessment (Without Ground Truth)', 
                     fontsize=18, fontweight='bold', y=0.998)
        
        # Plot 1: Distribution of reliability score
        axes[0, 0].hist(df_result['reliability_score'], bins=20, 
                       color='#2ecc71', edgecolor='black', alpha=0.7)
        axes[0, 0].set_xlabel('Reliability Score', fontsize=12)
        axes[0, 0].set_ylabel('Count', fontsize=12)
        axes[0, 0].set_title('Distribution of Reliability Scores', 
                            fontsize=14, fontweight='bold')
        axes[0, 0].grid(axis='y', alpha=0.3)
        axes[0, 0].axvline(70, color='red', linestyle='--', label='Threshold: 70')
        axes[0, 0].legend()
        
        # Plot 2: Score vs Plate Length
        axes[0, 1].scatter(df_result['plate_length'], 
                          df_result['reliability_score'],
                          alpha=0.6, c='#3498db', edgecolors='black', s=50)
        axes[0, 1].set_xlabel('Plate Length', fontsize=12)
        axes[0, 1].set_ylabel('Reliability Score', fontsize=12)
        axes[0, 1].set_title('Reliability vs Plate Length', 
                            fontsize=14, fontweight='bold')
        axes[0, 1].grid(alpha=0.3)
        
        # Plot 3: Score vs Aspect Ratio
        axes[0, 2].scatter(df_result['aspect_ratio'], 
                          df_result['reliability_score'],
                          alpha=0.6, c='#e74c3c', edgecolors='black', s=50)
        axes[0, 2].set_xlabel('Aspect Ratio', fontsize=12)
        axes[0, 2].set_ylabel('Reliability Score', fontsize=12)
        axes[0, 2].set_title('Reliability vs Aspect Ratio', 
                            fontsize=14, fontweight='bold')
        axes[0, 2].grid(alpha=0.3)
        
        # Plot 4: Issues distribution
        if all_issues:
            issue_labels = list(issue_counts.keys())[:8]
            issue_values = list(issue_counts.values())[:8]
            
            axes[1, 0].barh(issue_labels[::-1], issue_values[::-1],
                           color='#f39c12', edgecolor='black', alpha=0.8)
            axes[1, 0].set_xlabel('Count', fontsize=12)
            axes[1, 0].set_title('Top Issues Detected', 
                                fontsize=14, fontweight='bold')
            axes[1, 0].grid(axis='x', alpha=0.3)
        
        # Plot 5: Plate length distribution
        axes[1, 1].hist(df_result['plate_length'], bins=range(1, 15),
                       color='#9b59b6', edgecolor='black', alpha=0.7)
        axes[1, 1].set_xlabel('Plate Length', fontsize=12)
        axes[1, 1].set_ylabel('Count', fontsize=12)
        axes[1, 1].set_title('Plate Length Distribution', 
                            fontsize=14, fontweight='bold')
        axes[1, 1].grid(axis='y', alpha=0.3)
        
        # Plot 6: High vs Low confidence
        axes[1, 2].pie([len(high_conf), len(medium_conf), len(low_conf)], 
                      labels=[f'High (≥80)\n{len(high_conf)}', 
                             f'Medium (60-79)\n{len(medium_conf)}', 
                             f'Low (<60)\n{len(low_conf)}'],
                      colors=['#2ecc71', '#f39c12', '#e74c3c'],
                      autopct='%1.1f%%', startangle=90)
        axes[1, 2].set_title('Confidence Distribution', 
                            fontsize=14, fontweight='bold')
        
        plt.tight_layout(rect=[0, 0, 1, 0.995])
        plt.savefig(f'{output_dir}/quality_assessment.png', dpi=150, bbox_inches='tight')
        print(f"   ✅ Đã lưu biểu đồ: {output_dir}/quality_assessment.png")
        plt.show()
        
    except Exception as e:
        print(f"   ⚠️  Lỗi khi tạo biểu đồ: {e}")
    
    # Export kết quả
    print("\n💾 Đang export kết quả...")
    
    reliability_csv = f'{output_dir}/reliability_assessment.csv'
    df_result.to_csv(reliability_csv, index=False)
    print(f"   ✅ File đánh giá: {reliability_csv}")
    
    suspicious = df_result[df_result['reliability_score'] < 50]
    if len(suspicious) > 0:
        suspicious_csv = f'{output_dir}/suspicious_predictions.csv'
        suspicious.to_csv(suspicious_csv, index=False)
        print(f"   ✅ Predictions đáng ngờ (score < 50): {suspicious_csv}")
        print(f"      → {len(suspicious)} predictions")
    
    high_quality = df_result[df_result['reliability_score'] >= 80]
    if len(high_quality) > 0:
        high_quality_csv = f'{output_dir}/high_quality_predictions.csv'
        high_quality.to_csv(high_quality_csv, index=False)
        print(f"   ✅ Predictions chất lượng cao (score ≥ 80): {high_quality_csv}")
        print(f"      → {len(high_quality)} predictions")
    
    print("\n" + "="*80)
    print("✅ HOÀN TẤT ĐÁNH GIÁ!")
    print(f"📁 Tất cả kết quả đã lưu tại: {output_dir}")
    print("="*80)
    
    return df_result


if __name__ == "__main__":
    CSV_PATH = "/kaggle/working/license_plate_results.csv"
    IMAGE_DIR = "/kaggle/input/datasets/vdt1501/license-plate-detection-dataset/License Plate Detection Dataset/images/test"
    OUTPUT_DIR = "/kaggle/working/eval_no_gt"
    
    df_result = evaluate_ocr_no_gt(CSV_PATH, IMAGE_DIR, OUTPUT_DIR)

Writing evaluate_no_gt.py


Cell: Phân Tích Sâu Kết Quả Đánh Giá

In [5]:
%%writefile analyze_deep.py
import os
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

def deep_analysis(output_dir='/kaggle/working/eval_no_gt'):
    """Phân tích sâu kết quả đánh giá."""
    
    reliability_csv = f'{output_dir}/reliability_assessment.csv'
    
    if not os.path.exists(reliability_csv):
        print(f"❌ Không tìm thấy file: {reliability_csv}")
        print("   Hãy chạy Cell 1 trước!")
        return None
    
    df = pd.read_csv(reliability_csv)
    
    print("="*80)
    print("🔍 PHÂN TÍCH SÂU KẾT QUẢ ĐÁNH GIÁ")
    print("="*80)
    
    # Chia thành 3 nhóm
    high_conf = df[df['reliability_score'] >= 80]
    medium_conf = df[(df['reliability_score'] >= 60) & (df['reliability_score'] < 80)]
    low_conf = df[df['reliability_score'] < 60]
    
    print("\n📊 PHÂN TÍCH ĐỘ TIN CẬY:")
    print("-"*80)
    print(f"  • High confidence (≥80): {len(high_conf):4d} samples ({len(high_conf)/len(df)*100:5.1f}%)")
    print(f"  • Medium confidence (60-79): {len(medium_conf):4d} samples ({len(medium_conf)/len(df)*100:5.1f}%)")
    print(f"  • Low confidence (<60): {len(low_conf):4d} samples ({len(low_conf)/len(df)*100:5.1f}%)")
    
    # Phân tích issues
    print("\n⚠️  PHÂN TÍCH CÁC VẤN ĐỀ:")
    print("-"*80)
    
    all_issues = []
    for issues in df['issues']:
        if issues != 'None':
            all_issues.extend([i.strip() for i in issues.split(',')])
    
    if all_issues:
        issue_counts = Counter(all_issues)
        
        print("  Top 10 vấn đề phổ biến nhất:")
        for i, (issue, count) in enumerate(issue_counts.most_common(10), 1):
            pct = count / len(df) * 100
            print(f"    {i:2d}. {issue:30s}: {count:4d} lần ({pct:5.1f}%)")
    
    # Phân tích predictions đáng ngờ
    print("\n🚨 PHÂN TÍCH PREDICTIONS ĐÁNG NGỜ (Score < 50):")
    print("-"*80)
    
    suspicious = df[df['reliability_score'] < 50]
    
    if len(suspicious) > 0:
        print(f"  Tổng số predictions đáng ngờ: {len(suspicious)}")
        
        print(f"\n  📏 Độ dài plate:")
        print(f"    • Trung bình: {suspicious['plate_length'].mean():.1f} ký tự")
        if len(suspicious) > 0:
            print(f"    • Phổ biến nhất: {suspicious['plate_length'].mode()[0]} ký tự")
        
        print(f"\n  📐 Aspect ratio:")
        print(f"    • Trung bình: {suspicious['aspect_ratio'].mean():.2f}")
        print(f"    • Range: {suspicious['aspect_ratio'].min():.2f} - {suspicious['aspect_ratio'].max():.2f}")
        
        print(f"\n  🔴 Top 10 predictions đáng ngờ nhất:")
        top_suspicious = suspicious.nsmallest(10, 'reliability_score')
        for idx, row in top_suspicious.iterrows():
            print(f"    • {row['image']:40s} | Score: {row['reliability_score']:3.0f} | "
                  f"Plate: '{row['plate']}' | Length: {row['plate_length']}")
    else:
        print("  ✅ Không có predictions đáng ngờ (score < 50)")
    
    # So sánh High vs Low
    print("\n📈 SO SÁNH HIGH vs LOW CONFIDENCE:")
    print("-"*80)
    
    if len(high_conf) > 0 and len(low_conf) > 0:
        metrics = ['plate_length', 'aspect_ratio', 'bbox_area']
        
        print(f"\n  {'Metric':<20s} | {'High Conf (≥80)':<20s} | {'Low Conf (<60)':<20s}")
        print("  " + "-"*70)
        
        for metric in metrics:
            high_mean = high_conf[metric].mean()
            low_mean = low_conf[metric].mean()
            print(f"  {metric:<20s} | {high_mean:>15.2f} | {low_mean:>15.2f}")
    
    # Trực quan hóa
    print("\n📊 Đang tạo biểu đồ phân tích sâu...")
    
    try:
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('📊 Deep Analysis: High vs Low Confidence Predictions', 
                     fontsize=16, fontweight='bold')
        
        # Plot 1: Plate length distribution
        if len(high_conf) > 0:
            axes[0, 0].hist(high_conf['plate_length'], bins=range(1, 15), 
                           alpha=0.6, label='High Confidence', color='#2ecc71', edgecolor='black')
        if len(low_conf) > 0:
            axes[0, 0].hist(low_conf['plate_length'], bins=range(1, 15), 
                           alpha=0.6, label='Low Confidence', color='#e74c3c', edgecolor='black')
        axes[0, 0].set_xlabel('Plate Length', fontsize=12)
        axes[0, 0].set_ylabel('Count', fontsize=12)
        axes[0, 0].set_title('Plate Length Distribution', fontsize=14, fontweight='bold')
        axes[0, 0].legend()
        axes[0, 0].grid(axis='y', alpha=0.3)
        
        # Plot 2: Aspect ratio distribution
        if len(high_conf) > 0:
            axes[0, 1].hist(high_conf['aspect_ratio'], bins=30, 
                           alpha=0.6, label='High Confidence', color='#2ecc71', edgecolor='black')
        if len(low_conf) > 0:
            axes[0, 1].hist(low_conf['aspect_ratio'], bins=30, 
                           alpha=0.6, label='Low Confidence', color='#e74c3c', edgecolor='black')
        axes[0, 1].set_xlabel('Aspect Ratio', fontsize=12)
        axes[0, 1].set_ylabel('Count', fontsize=12)
        axes[0, 1].set_title('Aspect Ratio Distribution', fontsize=14, fontweight='bold')
        axes[0, 1].legend()
        axes[0, 1].grid(axis='y', alpha=0.3)
        
        # Plot 3: BBox area distribution
        if len(high_conf) > 0:
            axes[1, 0].hist(high_conf['bbox_area'], bins=30, 
                           alpha=0.6, label='High Confidence', color='#2ecc71', edgecolor='black')
        if len(low_conf) > 0:
            axes[1, 0].hist(low_conf['bbox_area'], bins=30, 
                           alpha=0.6, label='Low Confidence', color='#e74c3c', edgecolor='black')
        axes[1, 0].set_xlabel('BBox Area (pixels²)', fontsize=12)
        axes[1, 0].set_ylabel('Count', fontsize=12)
        axes[1, 0].set_title('BBox Area Distribution', fontsize=14, fontweight='bold')
        axes[1, 0].legend()
        axes[1, 0].grid(axis='y', alpha=0.3)
        
        # Plot 4: Score vs Plate Length
        if len(high_conf) > 0:
            axes[1, 1].scatter(high_conf['plate_length'], high_conf['reliability_score'],
                              alpha=0.6, label='High Confidence', color='#2ecc71', 
                              edgecolors='black', s=50)
        if len(low_conf) > 0:
            axes[1, 1].scatter(low_conf['plate_length'], low_conf['reliability_score'],
                              alpha=0.6, label='Low Confidence', color='#e74c3c', 
                              edgecolors='black', s=50)
        axes[1, 1].set_xlabel('Plate Length', fontsize=12)
        axes[1, 1].set_ylabel('Reliability Score', fontsize=12)
        axes[1, 1].set_title('Score vs Plate Length', fontsize=14, fontweight='bold')
        axes[1, 1].legend()
        axes[1, 1].grid(alpha=0.3)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.savefig(f'{output_dir}/deep_analysis.png', dpi=150, bbox_inches='tight')
        print(f"   ✅ Đã lưu biểu đồ: {output_dir}/deep_analysis.png")
        plt.show()
        
    except Exception as e:
        print(f"   ⚠️  Lỗi khi tạo biểu đồ: {e}")
    
    # Đề xuất hành động
    print("\n💡 ĐỀ XUẤT HÀNH ĐỘNG TỐI ƯU HÓA:")
    print("="*80)
    
    if all_issues:
        top_issue = issue_counts.most_common(1)[0]
        issue_name, issue_count = top_issue
        issue_pct = issue_count / len(df) * 100
        
        print(f"\n  🎯 VẤN ĐỀ CHÍNH: {issue_name} ({issue_pct:.1f}% samples)")
        
        if 'short' in issue_name.lower():
            print("\n  📋 HÀNH ĐỘNG:")
            print("    1. Tăng confidence threshold của YOLO (hiện tại: 0.20)")
            print("       → Thử: conf=0.30 hoặc conf=0.40")
            print("    2. Kiểm tra xem model có detect đúng license plates không")
            
        elif 'long' in issue_name.lower():
            print("\n  📋 HÀNH ĐỘNG:")
            print("    1. Giảm confidence threshold của YOLO")
            print("    2. Thêm post-processing: giới hạn độ dài tối đa")
            
        elif 'aspect' in issue_name.lower():
            print("\n  📋 HÀNH ĐỘNG:")
            print("    1. Thêm filter dựa trên aspect ratio trong post-processing")
            print("    2. License plates thường có aspect ratio 2.5-6.0")
            
        elif 'small' in issue_name.lower() or 'large' in issue_name.lower():
            print("\n  📋 HÀNH ĐỘNG:")
            print("    1. Thêm filter dựa trên diện tích bbox")
            print("    2. Loại bỏ các bbox quá nhỏ (< 1000 pixels²)")
    
    print("\n  🔧 CẢI THIỆN CHUNG:")
    print("    1. Thử nghiệm với các confidence thresholds khác nhau")
    print("    2. Thêm data augmentation cho PARSEQ")
    print("    3. Fine-tune YOLO với dataset cụ thể")
    print("    4. Thêm post-processing rules")
    
    print("="*80)
    
    return df


if __name__ == "__main__":
    df = deep_analysis('/kaggle/working/eval_no_gt')

Writing analyze_deep.py


 Xem Ảnh Các Predictions Đáng Ngờ

In [6]:
%%writefile visualize_suspicious.py
import os
import cv2
import pandas as pd
import matplotlib.pyplot as plt

def visualize_suspicious_predictions(output_dir='/kaggle/working/eval_no_gt', 
                                    image_dir="/kaggle/input/datasets/vdt1501/license-plate-detection-dataset/License Plate Detection Dataset/images/test"):
    """Trực quan hóa các predictions đáng ngờ."""
    
    suspicious_csv = f'{output_dir}/suspicious_predictions.csv'
    
    if not os.path.exists(suspicious_csv):
        print("❌ Không tìm thấy file suspicious_predictions.csv")
        print("   Có thể không có predictions nào có score < 50")
        return
    
    df = pd.read_csv(suspicious_csv)
    
    if len(df) == 0:
        print("✅ Không có predictions đáng ngờ!")
        return
    
    print(f"🔍 Đang trực quan hóa {len(df)} predictions đáng ngờ...")
    
    # Tạo thư mục output
    vis_dir = f'{output_dir}/suspicious_visualizations'
    os.makedirs(vis_dir, exist_ok=True)
    
    # Hiển thị top 20 predictions đáng ngờ nhất
    top_suspicious = df.nsmallest(min(20, len(df)), 'reliability_score')
    
    n_rows = (len(top_suspicious) + 4) // 5  # 5 cột
    n_cols = min(5, len(top_suspicious))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    fig.suptitle('🚨 Top Suspicious Predictions', 
                 fontsize=18, fontweight='bold', y=0.998)
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for idx in range(n_rows * n_cols):
        row_idx = idx // n_cols
        col_idx = idx % n_cols
        
        ax = axes[row_idx, col_idx]
        
        if idx < len(top_suspicious):
            row = top_suspicious.iloc[idx]
            
            img_path = os.path.join(image_dir, row['image'])
            
            if os.path.exists(img_path):
                img = cv2.imread(img_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # Parse bbox
                try:
                    bbox_str = str(row['bbox'])
                    if bbox_str.startswith('['):
                        bbox = eval(bbox_str)
                    else:
                        bbox = [float(x) for x in bbox_str.strip('[]()').split(',')]
                    
                    if len(bbox) == 4:
                        x1, y1, x2, y2 = map(int, bbox)
                        cv2.rectangle(img, (x1, y1), (x2, y2), (255, 0, 0), 3)
                        
                        cv2.putText(img, f"Score: {row['reliability_score']:.0f}", 
                                   (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
                        cv2.putText(img, f"Pred: {row['plate']}", 
                                   (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                except:
                    pass
                
                ax.imshow(img)
                ax.set_title(f"{row['image'][:25]}...\nScore: {row['reliability_score']:.0f} | "
                            f"Plate: '{row['plate']}'",
                            fontsize=9, fontweight='bold', color='red')
            else:
                ax.text(0.5, 0.5, f"Image not found\n{row['image']}",
                       ha='center', va='center', transform=ax.transAxes,
                       fontsize=10, color='red')
        else:
            ax.axis('off')
        
        ax.axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.995])
    plt.savefig(f'{vis_dir}/suspicious_predictions.png', dpi=150, bbox_inches='tight')
    print(f"\n✅ Đã lưu visualization tại: {vis_dir}/suspicious_predictions.png")
    plt.show()
    
    # Lưu từng ảnh riêng biệt
    print(f"\n💾 Đang lưu từng ảnh riêng biệt...")
    saved_count = 0
    for idx, row in top_suspicious.iterrows():
        img_path = os.path.join(image_dir, row['image'])
        if os.path.exists(img_path):
            img = cv2.imread(img_path)
            
            try:
                bbox_str = str(row['bbox'])
                if bbox_str.startswith('['):
                    bbox = eval(bbox_str)
                else:
                    bbox = [float(x) for x in bbox_str.strip('[]()').split(',')]
                
                if len(bbox) == 4:
                    x1, y1, x2, y2 = map(int, bbox)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 3)
                    
                    cv2.putText(img, f"Score: {row['reliability_score']:.0f}", 
                               (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
                    cv2.putText(img, f"Pred: {row['plate']}", 
                               (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
                    
                    output_path = f"{vis_dir}/{row['image']}"
                    cv2.imwrite(output_path, img)
                    saved_count += 1
            except:
                pass
    
    print(f"✅ Đã lưu {saved_count} ảnh tại: {vis_dir}/")


if __name__ == "__main__":
    visualize_suspicious_predictions()

Writing visualize_suspicious.py


In [8]:
# Cell 1: Đánh giá
!python evaluate_no_gt.py

# Cell 2: Phân tích
!python analyze_deep.py

# Cell 3: Trực quan hóa
!python visualize_suspicious.py

🚀 BẮT ĐẦU ĐÁNH GIÁ OCR (KHÔNG CẦN GROUND TRUTH)

📂 Loading predictions từ: /kaggle/working/license_plate_results.csv
   ✅ Loaded 862 predictions
   ✅ 862 predictions có bbox hợp lệ

🔢 Đang tính reliability score...
   ✅ Đã tính score cho 862 predictions

📊 KẾT QUẢ ĐÁNH GIÁ

📈 TỔNG QUAN:
  • Tổng số predictions: 862
  • Điểm trung bình: 60.1/100
  • Điểm trung vị: 60.0/100

🎯 PHÂN LOẠI ĐỘ TIN CẬY:
  • High confidence (≥80):  163 (18.9%)
  • Medium confidence (60-79):  368 (42.7%)
  • Low confidence (<60):  331 (38.4%)

⚠️  CÁC VẤN ĐỀ PHỔ BIẾN:
    • Invalid characters            : 685 lần
    • Too square                    : 413 lần
    • Good length                   : 277 lần
    • Good aspect ratio             :  57 lần
    • Very small                    :  36 lần

📊 Đang tạo biểu đồ...
/kaggle/working/evaluate_no_gt.py:230: UserWarning: Glyph 128269 (\N{LEFT-POINTING MAGNIFYING GLASS}) missing from font(s) DejaVu Sans.
  plt.tight_layout(rect=[0, 0, 1, 0.995])
/kaggle/working/eval